## Titration with PyNS (tPyNS) Tutorial:
Authored by: Abdallah Alashqar (abdallah.j.alashqar@fau.de)

Authored on: 04.06.2025

Titration refers to the process of changing the amplitude intensity to find the stimulation threshold (the minimum intensity) able of activating a fiber. Activation here means producing a propagating action potential along the fiber length.

Convergence in the titration process is defined by reaching a percentage of change predefined by the user

In [ ]:
import numpy as np
import h5py
# add parent directory to path
import sys
import os

# Add parent directory to path to import pyns modules
current_dir = os.path.dirname(os.path.abspath('__file__'))
pyns_root = os.path.join(current_dir, '..')
sys.path.insert(0, pyns_root)

from pyns.utils import filter_axon_trajectories, create_single_pulse_waveform
from pyns.axon_models import MyelinatedAxon

import matplotlib.pyplot as plt

### Step 1: load the main resources for your simulation (field dict and axon fibers)
- This is the same step as in the first tutorial [PyNS_tutorial_0.ipynb](https://gitlab.rrze.fau.de/ProModell/pyns/-/blob/main/tutorials/PyNS_tutorial_0.ipynb).

In [ ]:
# Load the field dictionary from the test dataset
# Get pyns root directory
pyns_root_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Path to test dataset files
field_path = os.path.join(pyns_root_dir, 'src', 'pyns', 'test_dataset', 'lumbar-tSCS_cathode_T11-T12_anode_navel-sides_units_V_m_cropped.h5')
axons_path = os.path.join(pyns_root_dir, 'src', 'pyns', 'test_dataset', 'RightSoleusAxons_diams_from_Schalow1992_cropped.npy')

# Load field dictionary from HDF5 file
with h5py.File(field_path, 'r') as f:
    field_dict = {key: f[key][()] for key in f.keys()}

# Convert coordinates from m to um for consistency with axon coordinates
field_dict["x"] *= 1e6  # m to um
field_dict["y"] *= 1e6  # m to um
field_dict["z"] *= 1e6  # m to um

print(f"Field dictionary entries:")
for key in field_dict.keys():
    print(f"  {key}: shape={field_dict[key].shape}, dtype={field_dict[key].dtype}")

# Define spatial ranges for filtering axon trajectories (with 1mm safety margin)
x_range = [field_dict["x"].min() + 1000, field_dict["x"].max() - 1000]
y_range = [field_dict["y"].min() + 1000, field_dict["y"].max() - 1000]
z_range = [field_dict["z"].min() + 1000, field_dict["z"].max() - 1000]

# Load and filter axon trajectories
fibers_dict = np.load(axons_path, allow_pickle=True)[()]
print(f"\nNumber of fibers before filtering: {len(fibers_dict)}")

# Minimum axon length: fibers shorter than this are unlikely to show meaningful responses
min_axon_length = 40.0 * 1e3  # in micrometers
axon_dicts = filter_axon_trajectories(
    fibers_dict,
    x_range=x_range,
    y_range=y_range,
    z_range=z_range,
    min_axon_length=min_axon_length,
    rank=0,
)
print(f"Number of fibers after filtering: {len(axon_dicts)}")

--------------------------------------------------------
### Step 2: Define simulation settings
Simulation settings include general simulation parameters such as simulation duration and integration time-step, model-related parameters such as axon model variant and descritization method and stimulation-related parameters such as waveform.
Stimulus intensity is the parameter we will keep changing until the fiber is activated.

#### General parameters

In [ ]:
sim_dur = 5 # milliseconds
sim_dt = 0.005 # milliseconds

#### Model-related Parameters:

In [ ]:
model_variant = "Gaines"  # Options: 'Alashqar', 'Gaines', 'MRG'
param_fit_method="continuous" # for morphological parameter fitting: "continuous" or "discrete"

The model variant defined above is encoded into two parameters that the simulation pipeline gets, these are `model_type` and `tuned_flag`.
- `model_type` (str or None):
    - "sensory": for forcing sensory variant parameters in gaines and the tuned model.
    - "motor": for forcing motor variant parameters in gaines and the tuned model.
    - "mrg": for forcing MRG original parameters.
    - None: for letting the algorithm decide whether to enforce sensory or motor variant parameters from the axon name for our models and gaines models. If no keywords were found, the default is to use sensory variant parameters. Currently the following keywords are used:
        - Sensory fiber keywords: ["Sensory", "senory", "Aalpha", "Abeta", "\_DR\_", "\_DL\_"]
        - Motor fiber keywords: ["Motor", "motor", "\_alpha", "\_VR\_", "\_VL\_"]
- `tuned_flag` (boolean):
    - Whether to use the tuned model or not, when model_type is not 'MRG'

In [ ]:
# This is the same code used in the simulation scripts that you will use later.
# Encode model_variant into model_type and tuned_flag parameters
if model_variant.lower() == "alashqar":
    # MyelinatedAxon class defines the model type based on the axon name (should contain 'sensory', 'motor', Aalpha', 'alpha', 'Abeta')
    model_type = None
    tuned_flag = True
elif model_variant.lower() == "gaines":
    model_type = None
    tuned_flag = False
elif model_variant.lower() == "mrg":
    model_type = "MRG"
    tuned_flag = False
else:
    raise ValueError(
        f"Invalid model_variant argument: {model_variant}. Accepted arguments: 'Alashqar', 'Gaines', 'MRG'"
    )

#### Stimulation parameters
here we can define our waveform using parameters such as pulse_width, pulse amplitude, pulse frequency, carrier frequency, ...etc. Helping functions are available under [utils.py](https://gitlab.rrze.fau.de/ProModell/pyns/-/blob/main/utils.py) for creating different kind of waveforms:
- For single-pulse waveforms: use `create_single_pulse()`
- For muti-pulse waveforms having user-defined onsets: use `create_multiple_pulses()`
- For high-frequency waveforms with the possibility of having a carrier frequency: use `create_highfreq_pulse()`

The following example creates a monophasic single-pulse waveform to be used in the simulation

In [ ]:
# it is recommended to use only 1 or -1 for the amplitude since this is intended to only define the polarity of the stimulation.
# Later we will be using another parameter to define the intensity of the stimulation, which would be scaling the defined waveform.
amplitude = 1.0
start_at = 1.0 # milliseconds
pulse_width = 2.0 # milliseconds
end_at = start_at + pulse_width # milliseconds
pulse_shape = "monophasic" # can be "monophasic" or "biphasic"
pulse_biphasic_flag = True if pulse_shape == "biphasic" else False

stim_pulse_t, stim_pulse = create_single_pulse_waveform(
    start_at=start_at,
    end_at=end_at,
    amplitude=amplitude,
    biphasic=pulse_biphasic_flag,
)

# now plot the stimulation pulse
plt.figure(figsize=(10, 4))
plt.plot(stim_pulse_t, stim_pulse, label=f"{pulse_shape} pulse", color='red')
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude (a.u.)")
plt.title("Stimulation Pulse")
plt.show()
plt.close()


#### Step 3: Run titraion
For this we will make use of the helping functions in titration_utils.py and simulation_utils.py

In [ ]:
import pyns.titration_utils
from importlib import reload
reload(pyns.titration_utils)
from pyns.titration_utils import *

In [ ]:
# choose an axon from the filtered axon dictionary
axon_info = axon_dicts[0]
# you can also choose another random axon to see differences in behavior
# axon_info = np.random.choice(axon_dicts)
print(f"Selected axon: {axon_info['axon_name']}\n with diameter: {axon_info['diam']} um, and length: {axon_info['length']*1e-3} mm.")

In [ ]:
stim_threshold = titrate_axon(
    axon_info=axon_info,
    model_type=model_type,
    field_dict=field_dict,
    stim_pulse=stim_pulse,
    dt=sim_dt,
    sim_dur=sim_dur,
    tuned_flag=tuned_flag,
    paramfit_method=param_fit_method,
    passive_end_nodes=False,
    exclude_end_node_flag=True,
    verbose=True,
    titration_conv_perc=1.0, # in percentage: 1%
    initial_increment_factor=1.5,
    max_stim_factor=3000.0, # titration stops if the stimulation factor exceeds this value
    initial_reduction_factor=0.5,
)

#### Step 4: Titrate multiple fibers with MPI support (HPC compliant)

Please refer to the README to run the an example on the provided test dataset with the command `pyns-run-titration`

In [ ]:
!pyns-run-titrations --help